In [7]:
import os
import commons as c

import pandas as pd
from scipy.stats import chi2_contingency, kruskal, friedmanchisquare, wilcoxon

# Merge and save DFs for equiv, normal and balanced

In [8]:
output_folder = 'results'
os.makedirs(output_folder, exist_ok=True)

for mutant_type in c.mutant_types:
    dataframes = []
    for threshold in c.thresholds:
        for hw in c.hardware:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            df['hardware'] = hw
            df['threshold'] = threshold
            dataframes.append(df)
    
    complete_df = pd.concat(dataframes, ignore_index=True)
    selected_columns = complete_df[['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Input', 'Input_type', 'Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Gate_type', 'Relative_position', 'Output_type', 'hardware', 'threshold']]
        
    new_rows = []

    for metric in c.metrics:
        # Extract true and predicted labels for the current metric
        true_labels = complete_df[f'Killed_I{metric}']
        predicted_labels = complete_df[f'Killed_N{metric}']
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric  
        metric_df['expected'] = true_labels  
        metric_df['predicted'] = predicted_labels  
        metric_df['correctness'] = (true_labels == predicted_labels)  
        new_rows.append(metric_df)
    
    metric_df = pd.concat(new_rows, ignore_index=True)
    metric_df.reset_index(drop=True, inplace=True)

    output_path = os.path.join(output_folder, f'results_{mutant_type}_complete.csv')
    metric_df.to_csv(output_path, index=False)   


# Statistical Analysis

In [9]:
csv_path = f'results/results_balanced_selected.csv'
df_balanced = pd.read_csv(csv_path, dtype={'threshold': str})

csv_path = f'results/results_equiv_selected.csv'
df_equiv = pd.read_csv(csv_path, dtype={'threshold': str})

csv_path = f'results/results_normal_selected.csv'
df_normal = pd.read_csv(csv_path, dtype={'threshold': str})

df = pd.concat([df_equiv, df_normal], ignore_index=True)

In [10]:
df['threshold_metric'] = list(zip(df['threshold'], df['metric']))

# Separate categorical and numerical columns
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for cat in ['threshold', 'metric', 'threshold_metric']:
    categorical_columns.remove(cat)  

print(categorical_columns)
print(numerical_columns)

['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Qubits_number', 'Position', 'Qubits', 'distance']


In [11]:
def ratio_correct_counts(df, col):
    if 'threshold_metric' not in df.columns or 'correctness' not in df.columns:
        raise ValueError("The DataFrame must contain 'threshold_metric' and 'correctness' columns.")
    
    # Step 1: Group by number of qubits and threshold metric, then count the correct values
    total_counts = (
        df.groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='total_count')  # Total count for each group
    )
    
    correct_counts = (
        df[df['correctness'] == True]  # Filter only correct rows
        .groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='correct_count')  # Reset index and name the count column
    )
    
    correct_counts = pd.merge(correct_counts, total_counts, on=[col, 'threshold_metric'], how='left')
    
    # Step 4: Calculate the ratio of correct_count to total_count
    correct_counts['correct_ratio'] = correct_counts['correct_count'] / correct_counts['total_count']
    
    return correct_counts

### Friedman Chi Square test

In [12]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in categorical_columns: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    print(grouped_ranks)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        stat, p_value = friedmanchisquare(*grouped_ranks)
        print(f"P-value from Friedman test: {p_value}")
        print(f"Stat from Friedman test: {stat}")
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

For category Input:
Input
PureState_0     [1.0, 0.5831399845320959, 0.6790409899458624, ...
PureState_1     [1.0, 0.5653518948182521, 0.6707914410930652, ...
PureState_10    [1.0, 0.5909785932721713, 0.6743119266055045, ...
PureState_11    [1.0, 0.5646024464831805, 0.6750764525993884, ...
PureState_12    [1.0, 0.6005351681957186, 0.6666666666666666, ...
                                      ...                        
Quratest_5      [1.0, 0.5506195225143548, 0.629193109700816, 0...
Quratest_6      [1.0, 0.6143850105772136, 0.5947416137805984, ...
Quratest_7      [1.0, 0.6334239951647024, 0.6174070716228468, ...
Quratest_8      [1.0, 0.5466360856269113, 0.6230886850152905, ...
Quratest_9      [1.0, 0.6311162079510704, 0.6185015290519877, ...
Name: correct_ratio, Length: 64, dtype: object
P-value from Friedman test: 3.2870618550787537e-16
Stat from Friedman test: 200.27351597685055
Result: Significant association between 'cat' and 'threshold_metric'.
For category Input_type:
Input_type


In [13]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in numerical_columns:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        
        try:
            stat, p_value = friedmanchisquare(*grouped_ranks)
            print(f"P-value from Friedman test: {p_value}")
            print(f"Stat from Friedman test: {stat}")
        except Exception as e:
            print(f'Error processing column {cat}: {str(e)}')
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

For category gates:
P-value from Friedman test: 3.1660258021705983e-08
Stat from Friedman test: 76.42608695652179
Result: Significant association between 'cat' and 'threshold_metric'.
For category depth:
P-value from Friedman test: 0.0009472854161843606
Stat from Friedman test: 36.280000000000015
Result: Significant association between 'cat' and 'threshold_metric'.
For category singlequbit_gates:
P-value from Friedman test: 0.00012389233453710994
Stat from Friedman test: 40.29523809523807
Result: Significant association between 'cat' and 'threshold_metric'.
For category multiqubit_gates:
P-value from Friedman test: 5.248717682812835e-07
Stat from Friedman test: 58.14117647058815
Result: Significant association between 'cat' and 'threshold_metric'.
For category Qubits_number:
P-value from Friedman test: 0.00018853633019760577
Stat from Friedman test: 22.133333333333347
Result: Significant association between 'cat' and 'threshold_metric'.
For category Position:
P-value from Friedman test

In [14]:
x = [72, 96, 88, 92, 74, 76, 82]
y = [120, 120, 132, 120, 101, 96, 112]
z = [76, 95, 104, 96, 84, 72, 76]
res = friedmanchisquare(x, y, z)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(x, y, z)
print("Kruskal")
print(res.pvalue)
print(res.statistic)
print('')


x = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
y = [1, 20, 30, 40, 50, 60, 70, 80, 90, 100]
z = [5, 25, 35, 45, 55, 65, 75, 85, 95, 105]

array = [x, y, z]
transposed_array = [[row[i] for row in array] for i in range(len(x))]

res = friedmanchisquare(*array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*transposed_array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

print('')
res = friedmanchisquare(*transposed_array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

Friedman
0.005063414171757498
10.57142857142857
Kruskal
0.0023531131300843075
12.10403218261764

Friedman
0.00034646043329340613
15.935483870967735
Kruskal
0.000700442550182995
28.799598751671862

Friedman
0.0013987676797964598
27.0
Kruskal
0.9232665759374912
0.15967454302274883


### Kruskal-Wallis test

In [ ]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']
significant = []
non_significant = []

for cat in all_cats: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    # Perform Kruskal-Wallis test
    stat, p_value = kruskal(*grouped_ranks)
    
    print(f"P-value from Kruskal-Wallis test: {p_value}")
    print(f"Stat from Kruskal-Wallis test: {stat}")
        
    if p_value < 0.05:
        print(f"Result: Significant association between {cat} and 'threshold_metric'.")
        significant.append(cat)
    else:
        print(f"Result: No significant association between {cat} and 'threshold_metric'.")
        non_significant.append(cat)
    print(f"================================================================")
    
    
print(f"Significant association between the threshold_metric selection and {significant}.")
print(f"Non-Significant association between the threshold_metric and {non_significant}.")
    